Neste projeto vamos trabalhar com dados reais e construir modelos de Machine Learning para tentar prever 9 diferentes tipos de ataques ou ameaças cibernéticas a dispositivos IoT.

Os  dados  foram  extraídos  do  site  abaixo
- https://research.unsw.edu.au/projects/unsw-nb15-dataset

Osdados do dataset UNSW-NB 15 foram criados pela ferramenta IXIA PerfectStorm no Cyber Range Lab daUNSW Canberra para gerar um híbrido de atividades normais modernas reais e comportamentos sintéticos de ataque contemporâneo. A ferramenta tcpdump foi utilizada para capturar 100 GB do tráfego bruto. Este conjunto de  dados  possui  nove  tipos  de  ataques,  a  saber,  Fuzzers,  Analysis,  Backdoors,  DoS,  Exploits, Generic, Reconnaissance, Shellcode e Worms. 

In [1]:
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Carrega os dados de treino
dados_treino = pd.read_csv('dados/UNSW_NB15_training.csv')
dados_treino.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


In [3]:
dados_treino.shape

(69603, 45)

In [5]:
dados_teste = pd.read_csv("dados/UNSW_NB15_testing.csv")
dados_teste.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,44230,0.000010,udp,-,INT,2,0,168,0,100000.002500,...,1,1,0,0,0,1,1,0,Reconnaissance,1
1,44231,0.420918,tcp,http,FIN,14,40,1478,48886,125.915265,...,1,1,0,0,1,2,1,0,Generic,1
2,44232,1.854257,tcp,http,FIN,60,16,68197,698,40.447468,...,1,1,0,0,0,2,1,0,Fuzzers,1
3,44233,1.848615,tcp,ftp,FIN,12,12,1594,682,12.441747,...,1,2,0,0,0,2,2,0,Fuzzers,1
4,44234,0.384483,tcp,-,FIN,10,6,1506,268,39.013427,...,1,2,0,0,0,3,6,0,Fuzzers,1


In [6]:
dados_teste.shape

(12729, 45)

### Análise Exploratória



In [7]:
dados_treino.isnull().sum()

id                   0
dur                  0
proto                0
service              0
state                0
spkts                0
dpkts                0
sbytes               0
dbytes               0
rate                 0
sttl                 0
dttl                 0
sload                0
dload                0
sloss                0
dloss                0
sinpkt               0
dinpkt               0
sjit                 0
djit                 0
swin                 0
stcpb                0
dtcpb                0
dwin                 0
tcprtt               0
synack               0
ackdat               0
smean                0
dmean                0
trans_depth          0
response_body_len    0
ct_srv_src           0
ct_state_ttl         0
ct_dst_ltm           0
ct_src_dport_ltm     0
ct_dst_sport_ltm     0
ct_dst_src_ltm       0
is_ftp_login         0
ct_ftp_cmd           0
ct_flw_http_mthd     0
ct_src_ltm           0
ct_srv_dst           0
is_sm_ips_ports      0
attack_cat 

In [9]:
# classe da variavel alvo 1 (categoria de ataque)
dados_treino.attack_cat.value_counts()

attack_cat
Normal            24667
Generic           18850
Exploits          10999
Fuzzers            5892
DoS                4075
Reconnaissance     3447
Analysis            677
Backdoor            583
Shellcode           370
Worms                43
Name: count, dtype: int64

In [10]:
# classe da variavel 2 (indica se foi acesso normal ou nao)
dados_treino.label.value_counts()

label
1    44936
0    24667
Name: count, dtype: int64

`Vamos trabalhar com a variavel 1, um problema de classificaçao multiclasse`

### Verificando balanceamento de classe

In [12]:
X_treino = dados_treino.drop(axis = 1, columns = ['attack_cat'])
X_treino = X_treino.drop(axis = 1, columns = ['label'])

In [13]:
y1_treino = dados_treino['attack_cat'].values 
y2_treino = dados_treino['label'].values

In [14]:
# Contagem de classes
classes, contagem = np.unique(y1_treino, return_counts = True)

# Exibir o resultado
for classe, count in zip(classes, contagem):
    print(f"Classe {classe}: {count} ocorrências")

Classe Analysis: 677 ocorrências
Classe Backdoor: 583 ocorrências
Classe DoS: 4075 ocorrências
Classe Exploits: 10999 ocorrências
Classe Fuzzers: 5892 ocorrências
Classe Generic: 18850 ocorrências
Classe Normal: 24667 ocorrências
Classe Reconnaissance: 3447 ocorrências
Classe Shellcode: 370 ocorrências
Classe Worms: 43 ocorrências


Temos um problema claro de desbalanceamento, se treinarmos assim, o modelo vai aprender mais sobre as classes que tem mais dados e menos para classes com menos dados

In [15]:
# Contagem de classes
classes, contagem = np.unique(y2_treino, return_counts = True)

# Exibir o resultado
for classe, count in zip(classes, contagem):
    print(f"Classe {classe}: {count} ocorrências")

Classe 0: 24667 ocorrências
Classe 1: 44936 ocorrências


In [16]:
# mesma separacao nos dados de teste 
X_teste = dados_teste.drop(axis = 1, columns = ['attack_cat'])
x_teste = X_teste.drop(axis = 1, columns = ['label'])

In [17]:
y1_teste = dados_teste['attack_cat'].values 
y2_teste = dados_teste['label'].values

In [18]:
np.unique(y1_teste)

array(['DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal',
       'Reconnaissance', 'Shellcode', 'Worms'], dtype=object)

In [19]:
np.unique(y2_teste)

array([0, 1])

In [20]:
np.unique(y1_treino)

array(['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic',
       'Normal', 'Reconnaissance', 'Shellcode', 'Worms'], dtype=object)

In [22]:
np.unique(y2_treino)

array([0, 1])

Nota: Podemos ter mais classes em treino do que em teste. Mas o contrário não podemos ter.

### Pré Processamento de dados de entrada

In [23]:
dados_treino.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


In [24]:
# Separando variáveis numéricas e categóricas
numerical_cols = X_treino.select_dtypes(include = ['int64', 'float64']).columns
categorical_cols = X_treino.select_dtypes(include = ['object', 'bool']).columns

In [25]:
numerical_cols

Index(['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl',
       'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit',
       'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat',
       'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src',
       'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd',
       'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports'],
      dtype='object')

In [26]:
categorical_cols

Index(['proto', 'service', 'state'], dtype='object')

Vamos aplicar as seguintes técnicas de pré-processamento:

- Para variáveis categóricas aplicamos One-Hot-Encoding.
- Para variáveis numéricas aplicamos a padronização.

In [28]:
# Define uma lista de transformações
transformacoes = [('ohe', OneHotEncoder(drop = 'first'), categorical_cols),
                  ('scale', StandardScaler(), numerical_cols)]

type(transformacoes)

list

In [29]:
# Criamos o objeto de transformação de colunas
transforma_colunas = ColumnTransformer(transformers = transformacoes)

# Fazemos o fit com dados de treino
transforma_colunas.fit(X_treino)

ColumnTransformer(transformers=[('ohe', OneHotEncoder(drop='first'),
                                 Index(['proto', 'service', 'state'], dtype='object')),
                                ('scale', StandardScaler(),
                                 Index(['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl',
       'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit',
       'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat',
       'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src',
       'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm',
       'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd',
       'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports'],
      dtype='object'))])

In [30]:
# Aplicamos as transformações em treino e teste
X_treino_transform = transforma_colunas.transform(X_treino)
X_teste_transform = transforma_colunas.transform(X_teste)

X_treino_transform.shape, X_teste_transform.shape

((69603, 188), (12729, 188))

## Pré-Processamento dos Dados de Saída

Vamos trabalhar com a variável alvo 1. Temos que transformar a variável para sua representação numérica. Usaremos Label Encoding.

In [31]:
pd.unique(y1_treino)

array(['Normal', 'Reconnaissance', 'Backdoor', 'DoS', 'Exploits',
       'Analysis', 'Fuzzers', 'Worms', 'Shellcode', 'Generic'],
      dtype=object)

In [32]:
# Define o objeto Label Encoder
label_enc = LabelEncoder()

# Faz o fit com dados de treino
label_enc.fit(y1_treino)

LabelEncoder()

In [33]:
# Aplicamos o encoder em dados de treino e teste
y1_treino_transform = label_enc.transform(y1_treino)
y1_teste_transform = label_enc.transform(y1_teste)

In [34]:
# Variável alvo transformada
np.unique(y1_treino_transform)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

## Modelagem Preditiva com Machine Learning

Vamos construir diferentes modelos, tratar o problema de desbalanceamento, comparar os modelos, escolher a melhor versão e fazer o deploy.

In [37]:
# Cria o modelo de classificação multiclasse com balanceamento de classes
clf = LogisticRegression(solver = 'lbfgs', 
                         random_state = 123, 
                         max_iter = 4000, 
                         multi_class = "ovr", 
                         class_weight = 'balanced')

Para lidar com o desbalanceamento das classes diretamente no modelo de aprendizado de máquina, uma abordagem comum é ajustar o peso de cada classe de acordo com sua frequência. Isso pode ser feito no modelo LogisticRegression usando o parâmetro class_weight. Esse parâmetro pode ser configurado para **'balanced'**, o que ajusta os pesos automaticamente com base na inversa da frequência das classes, ou você pode passar manualmente os pesos.

### Validação Cruzada Estratificada

O StratifiedKFold implementa a validação cruzada estratificada. A validação cruzada estratificada é uma variação da validação cruzada que busca garantir que a proporção das classes seja mantida em cada fold. Essa técnica é especialmente útil em casos de conjuntos de dados desequilibrados, onde algumas classes são muito menos frequentes do que outras. 

In [38]:
# Validação cruzada
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 123)

# Métricas de avaliação
scoring = ['accuracy', 'precision_micro', 'recall_micro', 'f1_micro', 'roc_auc_ovr']  

In [39]:
%%time
cv_results = cross_validate(estimator = clf,
                            X = X_treino_transform,
                            y = y1_treino_transform,
                            scoring = scoring,
                            cv = cv,
                            return_train_score = False) 

CPU times: user 22.1 s, sys: 144 ms, total: 22.2 s
Wall time: 22.3 s


In [40]:
cv_results

{'fit_time': array([4.39152884, 4.56523204, 4.29039478, 4.65172291, 4.27015495]),
 'score_time': array([0.02585721, 0.0252111 , 0.02623415, 0.02542019, 0.0252471 ]),
 'test_accuracy': array([0.77077796, 0.76309173, 0.77753035, 0.76472701, 0.76688218]),
 'test_precision_micro': array([0.77077796, 0.76309173, 0.77753035, 0.76472701, 0.76688218]),
 'test_recall_micro': array([0.77077796, 0.76309173, 0.77753035, 0.76472701, 0.76688218]),
 'test_f1_micro': array([0.77077796, 0.76309173, 0.77753035, 0.76472701, 0.76688218]),
 'test_roc_auc_ovr': array([0.95267201, 0.9499884 , 0.94956114, 0.9503889 , 0.95187894])}

In [41]:
cv_results['test_accuracy'].mean()

0.7686018477005302

`A Validação Cruzada Estratificada é uma técnica importante quando você está lidando com dados desbalanceados.`

Como Funciona Divisão em Folds: O conjunto de dados é dividido em \(K\) subconjuntos (folds).
- **Manutenção da Proporção:** Diferente do K-Folds tradicional, o método estratificado garante que cada fold mantenha a mesma proporção de classes que o conjunto de dados completo. Por exemplo, se o conjunto total tem 70% de registros "Masculino" e 30% "Feminino", cada fold também terá aproximadamente essa distribuição.
- **Treinamento e Teste:** O modelo é treinado em \(K-1\) folds e testado no fold restante. Este processo se repete \(K\) vezes, garantindo que cada fold seja usado como conjunto de teste uma vez.
- **Avaliação:** As métricas de desempenho de cada iteração são agregadas (geralmente calculando a média) para fornecer uma avaliação mais robusta e menos enviesada do modelo

Quando Usar:

- Quando seus dados têm classes desbalanceadas (uma ou mais classes são muito menos frequentes).
- Quando a preservação da distribuição de classes é crítica para a correta avaliação do modelo.
- Para garantir que o modelo aprenda a detectar as classes minoritárias de maneira eficaz.

### Avaliação do modelo

In [42]:
%%time
clf.fit(X = X_treino_transform, y = y1_treino_transform)

CPU times: user 5.89 s, sys: 29.2 ms, total: 5.92 s
Wall time: 5.94 s


LogisticRegression(class_weight='balanced', max_iter=4000, multi_class='ovr',
                   random_state=123)

In [43]:
# Previsões com dados de teste
y_pred_class = clf.predict(X = X_teste_transform)
y_pred_score = clf.predict_proba(X = X_teste_transform)[:, 1]

In [44]:
# Valores únicos das previsões de classe
np.unique(y_pred_class)

array([0, 1, 3, 4, 5, 6, 7, 8, 9])

In [45]:
# Valores únicos das previsões de probabilidade de classe
y_pred_score

array([9.58343089e-03, 6.19885944e-03, 1.75138803e-06, ...,
       1.94766504e-22, 1.94972438e-22, 4.45902481e-02])

In [46]:
# Valores únicos dos dados reais
np.unique(y1_teste_transform)

array([2, 3, 4, 5, 6, 7, 8, 9])

In [52]:
# Precision
precision_ontest = precision_score(y_true = y1_teste_transform, y_pred = y_pred_class, average = 'micro')
# Recall
recall_ontest = recall_score(y_true = y1_teste_transform, y_pred = y_pred_class, average = 'micro')
# Accuracy
accuracy_ontest = accuracy_score(y_true = y1_teste_transform, y_pred = y_pred_class)

In [53]:
# Resumindo
print('\nPrecision Score nos Dados de Teste: {:1.5f}'.format(precision_ontest))
print('\nRecall Score nos Dados de Teste: {:1.5f}'.format(recall_ontest))
print('\nAccuracy Score nos Dados de Teste: {:1.5f}'.format(recall_ontest))


Precision Score nos Dados de Teste: 0.80061

Recall Score nos Dados de Teste: 0.80061

Accuracy Score nos Dados de Teste: 0.80061


### Experimentando Diversos Modelos 

In [55]:
modelos = [
    ('LogisticRegression', LogisticRegression(random_state = 123, max_iter = 4000, multi_class = "ovr")),
    ('RandomForest', RandomForestClassifier(random_state = 123, class_weight = 'balanced')),
    ('DecisionTree', DecisionTreeClassifier(random_state = 123, class_weight = 'balanced'))
]

In [57]:
%%time

# Listas para as métricas
test_accuracy_list = []
test_precision_list = []
test_recall_list = []
test_f1_list = []

# Lista para armazenar o resultado do treino de cada modelo
model_names_list = []

print('\nIniciando o Treinamento. Aguarde...')

# Loop por cada modelo
for model_name, clf in modelos:
    
    print('\nTreinando o modelo:', model_name)

    # Fit
    clf.fit(X = X_treino_transform, 
            y = y1_treino_transform)
    
    # Previsão de classe com dados de teste
    y_pred_class = clf.predict(X = X_teste_transform)  
    
    # Previsão de probabilidade com dados de teste 
    y_pred_score = clf.predict_proba(X = X_teste_transform)[:, 1]

    # Acurácia em dados de teste
    accuracy_ontest = accuracy_score(y_true = y1_teste_transform, y_pred = y_pred_class)
    
    # Precision
    precision_ontest = precision_score(y_true = y1_teste_transform, 
                                       y_pred = y_pred_class, 
                                       average = 'micro')
    
    # Recall
    recall_ontest = recall_score(y_true = y1_teste_transform, 
                                 y_pred = y_pred_class, 
                                 average = 'micro')
    
    # F1 score
    f1_ontest = f1_score(y_true = y1_teste_transform, 
                         y_pred = y_pred_class, 
                         average = 'micro')

    # Armazena os resultados
    model_names_list.append(model_name)
    test_accuracy_list.append(accuracy_ontest)
    test_precision_list.append(precision_ontest)
    test_recall_list.append(recall_ontest)
    test_f1_list.append(f1_ontest)
    
    # Salvar o modelo no disco
    model_filename = f'modelos/{model_name}_modelo.pkl'
    joblib.dump(clf, model_filename)
    print(f'Modelo {model_name} salvo como {model_filename}')

print('\nTreinamento Concluído e Modelos Salvos no Disco.\n')


Iniciando o Treinamento. Aguarde...

Treinando o modelo: LogisticRegression
Modelo LogisticRegression salvo como modelos/LogisticRegression_modelo.pkl

Treinando o modelo: RandomForest
Modelo RandomForest salvo como modelos/RandomForest_modelo.pkl

Treinando o modelo: DecisionTree
Modelo DecisionTree salvo como modelos/DecisionTree_modelo.pkl

Treinamento Concluído e Modelos Salvos no Disco.

CPU times: user 1min 6s, sys: 347 ms, total: 1min 6s
Wall time: 1min 6s


In [67]:
# Cria um dicionário com os resultados
dict_resultado = {'Modelo': model_names_list,
                  'Acurácia': test_accuracy_list,
                  'Precisão': test_precision_list,
                  'Recall': test_recall_list,
                  'F1-Score': test_f1_list}

# Converte em dataframe
df_resultado = pd.DataFrame(dict_resultado)
# Visualiza o resultado
df_resultado.sort_values(by = 'F1-Score', ascending = False)

,Modelo,Acurácia,Precisão,Recall,F1-Score
1,RandomForest,0.973446,0.973446,0.973446,0.973446
0,LogisticRegression,0.967712,0.967712,0.967712,0.967712
2,DecisionTree,0.961348,0.961348,0.961348,0.961348


🧪 Métricas de Avaliação — Mini Resumo

✅ 1. Acurácia

O que é: Proporção de previsões corretas.

Quando usar: Quando as classes estão balanceadas.

Problema: Enganosa em datasets desbalanceados (ex.: 98% normal vs. 2% fraude).

✅ 2. Precisão (Precision)

O que é: Entre todas as previsões de positivo, quantas realmente são positivas.

Interpretação: “Se o modelo falou que é fraude, qual a chance de realmente ser fraude?”

Bom para: Minimizar falsos positivos.

✅ 3. Recall (Sensibilidade)

O que é: Entre todos os positivos reais, quantos o modelo encontrou.

Interpretação: “Quantas fraudes reais eu consigo capturar?”

Bom para: Minimizar falsos negativos (importante em detecção de fraude).

✅ 4. F1-Score

O que é: Média harmônica entre Precisão e Recall.

Por que usar: Perfeito quando o dataset é desbalanceado.

Equilíbrio: Penaliza mais se uma das métricas estiver muito baixa.

✅ 5. Matriz de Confusão

O que é: Tabela com:

True Positive (TP)

True Negative (TN)

False Positive (FP)

False Negative (FN)

Por que usar: Mostra onde o modelo está errando exatamente.

### Deploy e Uso do Melhor Modelo com Novos Dados

In [59]:
# Este registro representa ataque do tipo "Reconnaissance"
novo_acesso = pd.read_csv('dados/novo_acesso.csv')
# Aplica nos novos dados as mesmas transformações aplicadas em treino
novo_acesso_transform = transforma_colunas.transform(novo_acesso)

In [65]:
# Carrega o melhor modelo do disco - RandomForest
modelo_final = joblib.load('modelos/RandomForest_modelo.pkl')
# Faz a previsão
previsao = modelo_final.predict(X = novo_acesso_transform) 
# Classe prevista
previsao

array([7])

In [66]:
# Inverte o label encoding
previsao_label = label_enc.inverse_transform(previsao)
print(previsao_label)

['Reconnaissance']


- Acertô